# 03 — Merge & Prepare ML Dataset

**Goal:** Load the cleaned historical panel, define features/targets, split into train/test.  
**Reads:** `data/processed/historical_panel.csv`  
**Outputs:** `data/processed/train.csv`, `data/processed/test.csv`

Features (5): `gdp_per_capita`, `hdi`, `control_of_corruption`, `employment_agriculture`, `gini`  
Targets (3): `poverty_3`, `poverty_8_30`, `poverty_10`  
Primary target: `poverty_3`

## Imports and paths

In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Population excluded — GDP per capita already captures economic scale;
# raw population does not directly predict poverty rates.
FEATURES = ["gdp_per_capita", "hdi", "control_of_corruption", "employment_agriculture", "gini"]
TARGETS = ["poverty_3", "poverty_8_30", "poverty_10"]  # poverty_4_20 excluded (poverty gap data)
PRIMARY_TARGET = "poverty_3"

print(f"Features ({len(FEATURES)}): {FEATURES}")
print(f"Targets  ({len(TARGETS)}): {TARGETS}")
print(f"Primary target: {PRIMARY_TARGET}")

Features (5): ['gdp_per_capita', 'hdi', 'control_of_corruption', 'employment_agriculture', 'gini']
Targets  (3): ['poverty_3', 'poverty_8_30', 'poverty_10']
Primary target: poverty_3


## Load historical panel

In [26]:
panel = pd.read_csv(DATA_PROCESSED / "historical_panel.csv")
print(f"Raw panel: {panel.shape}")
print(f"Countries: {panel['country_code'].nunique()}, Years: {panel['year'].min()}–{panel['year'].max()}")
panel.head()

Raw panel: (6076, 14)
Countries: 217, Years: 1996–2023


,country_name,country_code,year,gdp_per_capita,hdi,control_of_corruption,employment_agriculture,gini,gini_observed,poverty_3,poverty_8_30,poverty_10,imputation_pct,target_missing_pct
0,Aruba,ABW,1996,27561.280907,0.6465,0.447887,26.144256,39.750,False,NaN,NaN,NaN,0.8,1.0
1,Aruba,ABW,1997,29102.894328,0.6450,0.449777,25.729159,39.750,False,NaN,NaN,NaN,0.8,1.0
2,Aruba,ABW,1998,29133.471545,0.6515,0.448097,25.015691,39.000,False,NaN,NaN,NaN,0.8,1.0
3,Aruba,ABW,1999,29056.996085,0.6580,0.454098,23.876970,38.675,False,NaN,NaN,NaN,0.8,1.0
4,Aruba,ABW,2000,31092.100762,0.6660,0.455592,23.290873,39.800,False,NaN,NaN,NaN,0.8,1.0


In [27]:
# Quick look at missing data before filtering
print("% NaN per column:")
print((panel[FEATURES + TARGETS].isna().mean() * 100).round(2))

% NaN per column:
gdp_per_capita             0.00
hdi                        0.00
control_of_corruption      0.00
employment_agriculture     0.00
gini                       0.00
poverty_3                 67.10
poverty_8_30              69.31
poverty_10                69.31
dtype: float64


## Filter rows

- Drop rows where **all** target columns are NaN (no poverty data at all)
- Drop rows where **all** feature columns are NaN

In [28]:
n_before = len(panel)
all_countries = set(panel["country_code"].unique())

# Drop rows where the PRIMARY target is NaN — we need real poverty observations to train on
panel = panel.dropna(subset=[PRIMARY_TARGET])
print(f"After dropping NaN {PRIMARY_TARGET}: {len(panel)} (removed {n_before - len(panel)})")

# Drop rows with ALL features NaN
n_before2 = len(panel)
panel = panel.dropna(subset=FEATURES, how="all")
print(f"After dropping all-NaN features: {len(panel)} (removed {n_before2 - len(panel)})")

print(f"\nFinal dataset: {panel.shape}")
print(f"Countries: {panel['country_code'].nunique()}")

# Show which countries were dropped (no poverty data at all)
dropped = sorted(all_countries - set(panel["country_code"].unique()))
print(f"\nCountries dropped ({len(dropped)}): {dropped}")

# Imputation quality of retained data
print(f"\nImputation % in retained data: "
      f"mean={panel['imputation_pct'].mean():.1f}%, "
      f"median={panel['imputation_pct'].median():.1f}%, "
      f"max={panel['imputation_pct'].max():.1f}%")

After dropping NaN poverty_3: 1999 (removed 4077)
After dropping all-NaN features: 1999 (removed 0)

Final dataset: (1999, 14)
Countries: 170

Countries dropped (47): ['ABW', 'AFG', 'AND', 'ASM', 'ATG', 'BHR', 'BHS', 'BMU', 'BRN', 'CHI', 'CUB', 'CUW', 'CYM', 'DMA', 'ERI', 'FRO', 'GIB', 'GRL', 'GUM', 'HKG', 'IMN', 'KHM', 'KNA', 'KWT', 'LBY', 'LIE', 'MAC', 'MAF', 'MCO', 'MNP', 'NCL', 'NZL', 'OMN', 'PLW', 'PRI', 'PRK', 'PYF', 'SAU', 'SGP', 'SMR', 'SOM', 'SXM', 'TCA', 'TTO', 'VCT', 'VGB', 'VIR']

Imputation % in retained data: mean=0.0%, median=0.0%, max=0.6%


### Check primary target coverage

In [29]:
for t in TARGETS:
    n_valid = panel[t].notna().sum()
    print(f"{t}: {n_valid} non-null rows ({n_valid/len(panel)*100:.1f}%)")

print(f"\nRows with valid {PRIMARY_TARGET}: {panel[PRIMARY_TARGET].notna().sum()}")

poverty_3: 1999 non-null rows (100.0%)
poverty_8_30: 1865 non-null rows (93.3%)
poverty_10: 1865 non-null rows (93.3%)

Rows with valid poverty_3: 1999


## Train / Test split

80/20 country-level split — each country appears entirely in train OR test, never in both.


In [30]:
# Country-level split: alle Zeilen eines Landes gehen komplett in Train ODER Test
countries = panel["country_code"].unique()

train_countries, test_countries = train_test_split(
    countries,
    test_size=0.2,
    random_state=42
)

train = panel[panel["country_code"].isin(train_countries)].reset_index(drop=True)
test  = panel[panel["country_code"].isin(test_countries)].reset_index(drop=True)

print(f"\nTrain: {train.shape} — {train['country_code'].nunique()} countries")
print(f"Test:  {test.shape}  — {test['country_code'].nunique()} countries")

assert len(set(train["country_code"]) & set(test["country_code"])) == 0
print("No overlap")


Train: (1626, 14) — 136 countries
Test:  (373, 14)  — 34 countries
No overlap


### Verify the split looks reasonable

In [31]:
print("=== Train set ===")
print(f"Year range: {train['year'].min()}–{train['year'].max()}")
print(f"{PRIMARY_TARGET} — mean: {train[PRIMARY_TARGET].mean():.2f}, "
      f"median: {train[PRIMARY_TARGET].median():.2f}, "
      f"non-null: {train[PRIMARY_TARGET].notna().sum()}")

print(f"\n=== Test set ===")
print(f"Year range: {test['year'].min()}–{test['year'].max()}")
print(f"{PRIMARY_TARGET} — mean: {test[PRIMARY_TARGET].mean():.2f}, "
      f"median: {test[PRIMARY_TARGET].median():.2f}, "
      f"non-null: {test[PRIMARY_TARGET].notna().sum()}")

print(f"\n=== Feature stats (train) ===")
print(train[FEATURES].describe().round(3))

=== Train set ===
Year range: 1996–2023
poverty_3 — mean: 10.50, median: 1.40, non-null: 1626

=== Test set ===
Year range: 1996–2023
poverty_3 — mean: 9.86, median: 3.80, non-null: 373

=== Feature stats (train) ===
       gdp_per_capita       hdi  control_of_corruption  \
count        1626.000  1626.000               1626.000   
mean        22656.105     0.762                  0.544   
std         20822.904     0.141                  0.213   
min           365.201     0.263                  0.148   
25%          7026.112     0.684                  0.370   
50%         16267.051     0.785                  0.483   
75%         33001.939     0.879                  0.711   
max        160210.315     0.975                  0.992   

       employment_agriculture      gini  
count                1626.000  1626.000  
mean                   20.412    36.615  
std                    20.084     7.936  
min                     0.571    23.700  
25%                     4.028    30.700  
50%     

## Save train and test sets

In [32]:
train.to_csv(DATA_PROCESSED / "train.csv", index=False)
test.to_csv(DATA_PROCESSED / "test.csv", index=False)

print(f"Saved: {DATA_PROCESSED / 'train.csv'} — {train.shape}")
print(f"Saved: {DATA_PROCESSED / 'test.csv'}  — {test.shape}")

Saved: ../data/processed/train.csv — (1626, 14)
Saved: ../data/processed/test.csv  — (373, 14)


## Note on scaling

Tree-based models (XGBoost, LightGBM, Random Forest) do **not** need feature scaling.  
For Ridge regression and MLP, we will apply `StandardScaler` in the training notebook,  
fit on training data only to avoid data leakage.

---
**Outputs produced:**
- `data/processed/train.csv` — 80% of data, all columns preserved
- `data/processed/test.csv` — 20% of data, all columns preserved

Both files contain features, all 3 poverty targets, metadata (country, year), and `imputation_pct`.